In [ ]:
# imports to enable the notebook to access the src folder
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))

# import the necessary packages
from src.molecules import StructuralValidator
from src.pure import PropertyCalculator
from src.pure import AntoineCalculator
from src.data import load_unifac_data
from src.mixture.unifac.pyomo_iss import pyo_iss
import numpy as np
import matplotlib.pyplot as plt

In [28]:
# step1: define the compounds
mixture = {'acetone': {'CH3':1, 'CH3CO':1},
           'methanol': {'CH3OH':1},
           #'water': {'H2O':1}
           'DOH': {'DOH':1}
           }
# validate the structures
validator = StructuralValidator()
validator.validate(mixture)

{'acetone': True, 'methanol': True, 'DOH': True}

In [29]:
# step2: calculate the required properties
prop_list = ['tc', 'pc', 'tb', 'ac']
pure_calc = PropertyCalculator()
pure_props = pure_calc.calculate(mixture, prop_list)
# fix the properties of acetone
pure_props['acetone']['tc'] = 508.1
pure_props['acetone']['pc'] = 48.0
pure_props['acetone']['tb'] = 329.3
pure_props

{'acetone': {'tc': 508.1, 'pc': 48.0, 'tb': 329.3, 'ac': 0.33226923286882887},
 'methanol': {'tc': 513.0000000000005,
  'pc': 80.99999999999997,
  'tb': 337.84999999999985,
  'ac': 0.5639909999999987},
 'DOH': {'tc': 719.7000000000002,
  'pc': 85.38000000000001,
  'tb': 470.44999999999993,
  'ac': 0.4868299999999998}}

In [30]:
boiling_points = [props['tb'] for props in pure_props.values()]
T_min = min(boiling_points) 
T_max = max(boiling_points) 

In [31]:
# step3: calculate the Antoine coefficients
antoine_params = {}
antoine_calc = AntoineCalculator()
for compound, props in pure_props.items():
    results = antoine_calc.calculate_antoine_parameters(mixture, critical_properties=props)
    print('optimization results was', results['termination_condition'] )
    antoine_params[compound] = {}
    antoine_params[compound]['A'] = results['A']
    antoine_params[compound]['B'] = results['B']
    antoine_params[compound]['C'] = results['C']
antoine_params

optimization results was optimal
optimization results was optimal
optimization results was optimal


{'acetone': {'A': 5.105487152022129,
  'B': 1863.5123768521207,
  'C': 36.111024354679266},
 'methanol': {'A': 5.253881516250653,
  'B': 1616.1411202945164,
  'C': -29.905956718707568},
 'DOH': {'A': 4.044495853167801,
  'B': 1104.6851035245213,
  'C': -196.9304442942847}}

In [32]:
# Helper function to create initialization dictionaries
def create_init_dict_1d(values, fallback):
    """Create 1D initialization dict"""
    if init_values is not None and values is not None:
        return {i: float(values[i]) for i in range(len(values))}
    return fallback

def create_init_dict_2d(values, fallback, keys1=None, keys2=None):
    """Create 2D initialization dict with optional custom keys"""
    if init_values is not None and values is not None:
        values = np.atleast_2d(values)  # Ensure 2D for single data point case
        if keys1 is not None and keys2 is not None:
            return {(k1, k2): float(values[i, j]) for i, k1 in enumerate(keys1) for j, k2 in enumerate(keys2)}
        else:
            return {(j, i): float(values[j, i]) for j in range(values.shape[0]) for i in range(values.shape[1])}
    return fallback

def create_init_dict_3d(values, fallback, keys1=None, keys2=None, keys3=None):
    """Create 3D initialization dict with optional custom keys"""
    if init_values is not None and values is not None:
        # Ensure at least 3D (handle single data point case)
        if values.ndim == 2:
            values = values[np.newaxis, :, :]  # Add data point dimension
        if keys1 is not None and keys2 is not None and keys3 is not None:
            return {(k1, k2, k3): float(values[i, j, k]) 
                    for i, k1 in enumerate(keys1) 
                    for j, k2 in enumerate(keys2) 
                    for k, k3 in enumerate(keys3)}
        else:
            return {(j, i, k): float(values[j, i, k]) for j in range(values.shape[0]) 
                    for i in range(values.shape[1]) for k in range(values.shape[2])}
    return fallback

In [33]:
#init_values = _generate_initialization_values(x0, np.array([350]), mixture, rk_data, qk_data, interaction_data, antoine_params)

In [34]:
#init_values['J']

In [35]:
#create_init_dict_2d(init_values['J'] if init_values else None, 1.13)

In [ ]:
# suggest initial values of x and T
T0 = np.array([(T_max + T_min) / 2, (T_max + T_min) / 2])  # Two temperatures for two data points
xA = 0.5
x_iso = 0.75
x0 = np.array([[x_iso, 0.0, 1-x_iso],    # Point 0: iso-volatility point
                [xA, 0.0, 1-xA]])          # Point 1: minimum point 
# load unifac data
rk_data, qk_data, interaction_data = load_unifac_data()
# perform the VLE calculation
results, model = pyo_iss(x0, T0, mixture, rk_data, qk_data, interaction_data, antoine_params, verbose=True)

Ipopt 3.14.13: halt_on_ampl_error=yes
max_iter=10000
tol=1e-08


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.13, running with linear solver MUMPS 5.2.1.

Number of nonzeros in equality constraint Jacobian...:      597
Number of nonzeros in inequality constraint Jacobian.:        2
Number of nonzeros in Lagrangian Hessian.............:      470

Total number of variables............................:      148
                     variables with only lower bounds:      118
                variables with lower and upper bounds:       12
                     variables with only upper bounds:        0
Total number of equality con

In [37]:
# retrieving the solutions
gamma = np.empty_like(x0)
T = np.empty_like(T0)
y = np.empty_like(x0)
x = np.empty_like(x0)
psat = np.empty_like(x0)


for i in model.gamma.keys():
    gamma[i] = model.gamma[i].value

for i in model.T.keys():
    T[i] = model.T[i].value

for i in model.y.keys():
    y[i] = model.y[i].value

for i in model.x.keys():
    x[i] = model.x[i].value

for i in model.psat.keys():
    psat[i] = model.psat[i].value


print(x)
print(model.obj())

[[ 8.65806817e-01 -4.53471039e-34  1.34193183e-01]
 [ 5.73087025e-01 -4.03573958e-43  4.26912975e-01]]
1.0309661128368919


In [38]:
obj_value = model.obj()
obj_value

1.0309661128368919

In [39]:
model.obj()

1.0309661128368919